In [1]:
from pathlib import Path
import sys

# This notebook is stored in <journal package>/notebooks.
notebooks_dir = Path.cwd()
if notebooks_dir.name != "notebooks":
    candidate = notebooks_dir / "notebooks"
    if candidate.exists():
        notebooks_dir = candidate

project_root = notebooks_dir.parent
nhanes_dir = project_root / "src" / "nhanes"
jordan_dir = project_root / "src" / "jordan_external_validation"
expected_nhanes = project_root / "outputs" / "nhanes"
expected_jordan = project_root / "outputs" / "jordan"

assert (project_root / "README.md").exists(), "Open this notebook from the journal package"

if str(nhanes_dir) not in sys.path:
    sys.path.insert(0, str(nhanes_dir))

print("Project root:", project_root)
print("Python executable:", sys.executable)


Project root: <project_root>
Python executable: python


In [2]:
import importlib.metadata as metadata
import json
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import revision_analysis as ra

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Python:", sys.version)
for package in ["numpy", "pandas", "scipy", "scikit-learn", "tensorflow", "shap"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


Python: 3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]
numpy: 2.4.4
pandas: 3.0.2
scipy: 1.17.1
scikit-learn: 1.8.0
tensorflow: 2.21.0
shap: 0.51.0


In [3]:
DOWNLOAD_NHANES = True

raw_dir = project_root / "data" / "raw"
raw_files = sorted(raw_dir.glob("*.xpt"))

print("Existing XPORT files:", len(raw_files))
for path in raw_files:
    print(" -", path.name)

if DOWNLOAD_NHANES:
    subprocess.run(
        [sys.executable, "download_sources.py"],
        cwd=nhanes_dir,
        check=True,
    )
    raw_files = sorted(raw_dir.glob("*.xpt"))

print("XPORT files after acquisition step:", len(raw_files))


Existing XPORT files: 14
 - ALQ_J.xpt
 - BMX_J.xpt
 - BPQ_J.xpt
 - BPX_J.xpt
 - DEMO_J.xpt
 - DIQ_J.xpt
 - GLU_J.xpt
 - HDL_J.xpt
 - MCQ_J.xpt
 - PAQ_J.xpt
 - SLQ_J.xpt
 - SMQ_J.xpt
 - TCHOL_J.xpt
 - TRIGLY_J.xpt
XPORT files after acquisition step: 14


In [4]:
RUN_RECONSTRUCTION_DEMONSTRATION = True

if RUN_RECONSTRUCTION_DEMONSTRATION:
    assert len(raw_files) == 14, "All 14 public XPORT files are required"

    # design.reconstruct() reads the merged prepared CSV. Build it from the
    # fourteen raw XPORT files when it is not already present.
    prepared_file = project_root / "data" / "prepared" / "NHANES_2017_2018_Merged.csv"
    if not prepared_file.exists():
        print("Prepared merged file is missing; running prepare_raw.py ...")
        subprocess.run(
            [sys.executable, "prepare_raw.py"],
            cwd=nhanes_dir,
            check=True,
        )

    assert prepared_file.exists(), "prepare_raw.py did not create the merged NHANES CSV"
    import design

    merged_data, adult_men_data = design.reconstruct()
    definite_outcome_data = adult_men_data.loc[
        adult_men_data["DIQ010"].isin([1, 2])
    ].copy()
    print("Merged participants:", len(merged_data))
    print("Adult men before outcome restriction:", len(adult_men_data))
    print("Definite DIQ010 responses:", len(definite_outcome_data))
    print(definite_outcome_data["DIQ010"].value_counts(dropna=False))
    display(definite_outcome_data.head())
else:
    print("Reconstruction demonstration skipped.")


Merged participants: 9254
Adult men before outcome restriction: 2840
Definite DIQ010 responses: 2753
DIQ010
2.0    2276
1.0     477
Name: count, dtype: int64


,SEQN,SDMVSTRA,SDMVPSU,WTMEC2YR,DIQ010,RIAGENDR,RIDAGEYR,RIDSTATR,BMXWAIST,BMXHIP,MCQ300C,age,bmi,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,poverty_income_ratio,WTSAF2YR,waist_to_hip_ratio,systolic_bp,diastolic_bp,sleep_hours_per_day,alcohol_drinks_per_drinking_day,physical_activity_minutes_per_week,ever_smoked_100_cigarettes,hypertension_history,family_history_diabetes,education_level,ethnicity,diabetes,psu_group
0,93706.0,134.0,2.0,8723.439814,2.0,1.0,18.0,2.0,79.3,94.4,NaN,18.0,21.5,82.0,148.0,47.0,NaN,NaN,NaN,NaN,NaN,0.840042,111.333333,73.333333,10.5,NaN,285.0,No,No,NaN,NaN,Non-Hispanic Asian,0,134_2
1,93711.0,134.0,2.0,12390.919724,2.0,1.0,56.0,2.0,86.6,90.7,2.0,56.0,21.3,62.0,238.0,72.0,156.0,48.0,107.0,5.00,29226.045506,0.954796,101.333333,66.666667,7.0,1.0,600.0,No,No,No,College graduate or above,Non-Hispanic Asian,0,134_2
2,93712.0,147.0,2.0,30336.654325,2.0,1.0,18.0,2.0,72.0,88.2,NaN,18.0,19.7,68.0,182.0,48.0,NaN,NaN,NaN,0.76,NaN,0.816327,113.333333,70.000000,7.5,NaN,4050.0,Yes,No,NaN,NaN,Mexican American,0,147_2
3,93713.0,140.0,1.0,166841.660681,2.0,1.0,67.0,2.0,99.7,91.0,2.0,67.0,23.5,90.0,184.0,48.0,NaN,NaN,NaN,2.65,NaN,1.095604,104.666667,72.000000,5.5,3.0,270.0,Yes,No,No,High school graduate/GED,Non-Hispanic White,0,140_1
4,93715.0,139.0,2.0,9250.904672,2.0,1.0,71.0,2.0,89.7,93.1,1.0,71.0,22.5,66.0,180.0,57.0,NaN,NaN,NaN,1.56,NaN,0.963480,114.000000,59.333333,5.0,NaN,6040.0,Yes,No,Yes,High school graduate/GED,Other/Multiracial,0,139_2


In [5]:
live_outputs = nhanes_dir / "outputs"
nhanes_results = (
    live_outputs
    if (live_outputs / "cohort_and_split_composition.csv").exists()
    else expected_nhanes
)

cohort = pd.read_csv(nhanes_results / "cohort_and_split_composition.csv")
display(cohort)

primary_all = cohort.query("panel == 'mec_nonfasting' and split == 'all'").iloc[0]
assert int(primary_all["n"]) == 2589
assert int(primary_all["positive"]) == 459
print("PASS: primary cohort contains 2,589 men and 459 positive reports.")


,panel,split,n,positive,nonpositive,borderline,psus,strata,weight_total,weighted_positive_proportion
0,mec_nonfasting,all,2589,459,2130,0,30,15,1.158530e+08,0.129290
1,mec_nonfasting,test,515,93,422,0,6,3,2.243015e+07,0.155660
2,mec_nonfasting,train,1551,271,1280,0,18,9,7.019660e+07,0.119338
3,mec_nonfasting,validation,523,95,428,0,6,3,2.322626e+07,0.133901
4,fasting_with_glucose,all,1096,213,883,0,30,15,1.171780e+08,0.126867
5,fasting_with_glucose,test,215,44,171,0,6,3,2.252354e+07,0.176346
6,fasting_with_glucose,train,654,124,530,0,18,9,7.066564e+07,0.102207
7,fasting_with_glucose,validation,227,45,182,0,6,3,2.398882e+07,0.153055
8,fasting_without_glucose,all,1096,213,883,0,30,15,1.171780e+08,0.126867
9,fasting_without_glucose,test,215,44,171,0,6,3,2.252354e+07,0.176346


PASS: primary cohort contains 2,589 men and 459 positive reports.


In [6]:
psu_assignment = pd.read_csv(nhanes_results / "psu_assignment.csv")

display(psu_assignment.head(10))
display(
    psu_assignment.groupby("split").agg(
        psus=("psu_group", "nunique"),
        strata=("SDMVSTRA", "nunique"),
    )
)

psu_overlap_check = psu_assignment.groupby("psu_group")["split"].nunique().max()
assert psu_overlap_check == 1

primary_splits = cohort.query("panel == 'mec_nonfasting' and split != 'all'")
display(primary_splits[["split", "n", "positive", "psus", "strata"]])

assert primary_splits.set_index("split").loc["train", "n"] == 1551
assert primary_splits.set_index("split").loc["validation", "n"] == 523
assert primary_splits.set_index("split").loc["test", "n"] == 515
print("PASS: frozen train/validation/test partitions have no PSU overlap.")


,SDMVSTRA,SDMVPSU,psu_group,split
0,134.0,1.0,134_1,train
1,134.0,2.0,134_2,train
2,135.0,1.0,135_1,test
3,135.0,2.0,135_2,test
4,136.0,1.0,136_1,train
5,136.0,2.0,136_2,train
6,137.0,1.0,137_1,train
7,137.0,2.0,137_2,train
8,138.0,1.0,138_1,train
9,138.0,2.0,138_2,train


,psus,strata
split,,
test,6,3
train,18,9
validation,6,3


,split,n,positive,psus,strata
1,test,515,93,6,3
2,train,1551,271,18,9
3,validation,523,95,6,3


PASS: frozen train/validation/test partitions have no PSU overlap.


In [7]:
primary_numeric = [
    variable for variable in ra.NUMERIC
    if variable not in ["glucose_fasting", "ldl_cholesterol", "triglycerides"]
]
categorical_features = list(ra.CATEGORICAL)

predictor_sets = {
    "BASE": [x for x in primary_numeric if x not in ["bmi", "waist_to_hip_ratio"]] + categorical_features,
    "BMI": [x for x in primary_numeric if x != "waist_to_hip_ratio"] + categorical_features,
    "WHR": [x for x in primary_numeric if x != "bmi"] + categorical_features,
    "BOTH": primary_numeric + categorical_features,
}

for configuration, predictors in predictor_sets.items():
    print(f"{configuration}: {len(predictors)} predictors")
    print(predictors)
    print()


BASE: 15 predictors
['age', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'physical_activity_minutes_per_week', 'alcohol_drinks_per_drinking_day', 'sleep_hours_per_day', 'poverty_income_ratio', 'ever_smoked_100_cigarettes', 'education_level', 'ethnicity', 'hypertension_history', 'family_history_diabetes']

BMI: 16 predictors
['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'physical_activity_minutes_per_week', 'alcohol_drinks_per_drinking_day', 'sleep_hours_per_day', 'poverty_income_ratio', 'ever_smoked_100_cigarettes', 'education_level', 'ethnicity', 'hypertension_history', 'family_history_diabetes']

WHR: 16 predictors
['age', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'physical_activity_minutes_per_week', 'alcohol_drinks_per_drinking_day', 'sleep_hours_per_day', 'poverty_income_ratio', 'ever_smoked_100_cigarettes', 'education_level

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_visible_preprocessor(numeric_variables, categorical_variables):
    numeric_pipeline = Pipeline([
        ("standardize", StandardScaler()),
        ("impute", KNNImputer(n_neighbors=5, keep_empty_features=True)),
    ])

    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
        ("encode", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_variables),
        ("categorical", categorical_pipeline, categorical_variables),
    ])


preprocessor_example = make_visible_preprocessor(primary_numeric, categorical_features)
print(preprocessor_example)


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('standardize',
                                                  StandardScaler()),
                                                 ('impute',
                                                  KNNImputer(keep_empty_features=True))]),
                                 ['age', 'bmi', 'waist_to_hip_ratio',
                                  'systolic_bp', 'diastolic_bp', 'heart_rate',
                                  'cholesterol_total', 'hdl_cholesterol',
                                  'physical_activity_minutes_per_week',
                                  'alcohol_drinks_per_drinking_day',
                                  'sleep_hours_per_day',
                                  'poverty_income_ratio']),
                                ('categorical',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(keep_empty_feature

In [9]:
RUN_VISIBLE_PRIMARY_MODELS = False

primary_data_file = live_outputs / "mec_nonfasting_analysis_data.csv"

if RUN_VISIBLE_PRIMARY_MODELS:
    assert primary_data_file.exists(), "Run the complete NHANES pipeline first"
    analysis_data = pd.read_csv(primary_data_file)

    train = analysis_data.query("split == 'train'").copy()
    validation = analysis_data.query("split == 'validation'").copy()
    test = analysis_data.query("split == 'test'").copy()

    visible_results = []
    for configuration, removed_variables in ra.CONFIGS.items():
        numeric_variables = [x for x in primary_numeric if x not in removed_variables]
        model_features = numeric_variables + categorical_features

        fitted_preprocessor, fitted_model = ra.fit_model(
            "LogReg",
            train[model_features],
            train["diabetes"],
            train["analysis_weight"],
            numeric_variables,
        )

        validation_score, score_kind = ra.score_model(
            fitted_preprocessor,
            fitted_model,
            validation[model_features],
        )
        calibrator = ra.calibrate(
            validation_score,
            validation["diabetes"],
            validation["analysis_weight"],
            score_kind,
        )

        test_score, _ = ra.score_model(
            fitted_preprocessor,
            fitted_model,
            test[model_features],
        )
        test_probability = ra.apply_calibrator(calibrator, test_score, score_kind)
        test_metrics = ra.metrics(
            test["diabetes"],
            test_probability,
            test["analysis_weight"],
            threshold=0.20,
        )
        visible_results.append({"configuration": configuration, **test_metrics})

    visible_results = pd.DataFrame(visible_results)
    display(visible_results[[
        "configuration", "auc", "brier", "sensitivity", "specificity",
        "calibration_in_the_large", "calibration_slope"
    ]].round(4))
else:
    print("Visible model demonstration skipped. Set RUN_VISIBLE_PRIMARY_MODELS=True after the full pipeline run.")


Visible model demonstration skipped. Set RUN_VISIBLE_PRIMARY_MODELS=True after the full pipeline run.


In [10]:
logistic_results = pd.read_csv(nhanes_results / "common_logistic_results.csv")
neural_results = pd.read_csv(nhanes_results / "all_neural_test_results.csv")
paired_results = pd.read_csv(nhanes_results / "paired_logistic_auc_differences.csv")

display(logistic_results[[
    "panel", "config", "auc", "brier", "sensitivity", "specificity",
    "accuracy", "calibration_in_the_large", "calibration_slope"
]].round(4))

display(neural_results.round(4))
display(paired_results.round(4))


,panel,config,auc,brier,sensitivity,specificity,accuracy,calibration_in_the_large,calibration_slope
0,mec_nonfasting,BASE,0.8884,0.0893,0.6979,0.8675,0.8411,0.1736,0.8796
1,mec_nonfasting,BMI,0.8833,0.0897,0.6960,0.8622,0.8364,0.1997,0.8704
2,mec_nonfasting,BOTH,0.8863,0.0866,0.6646,0.8837,0.8496,0.2043,0.8718
3,mec_nonfasting,WHR,0.8897,0.0857,0.6566,0.8815,0.8465,0.1932,0.8881
4,fasting_with_glucose,BASE,0.9032,0.0742,0.7316,0.8676,0.8436,0.0206,0.6561
5,fasting_with_glucose,BMI,0.9015,0.0751,0.7407,0.8962,0.8687,0.1709,0.6360
6,fasting_with_glucose,BOTH,0.9047,0.0715,0.7336,0.8858,0.8589,0.1178,0.7180
7,fasting_with_glucose,WHR,0.9067,0.0714,0.7245,0.8698,0.8442,0.0660,0.7317
8,fasting_without_glucose,BASE,0.8255,0.1006,0.7140,0.8603,0.8345,0.1693,0.6645
9,fasting_without_glucose,BMI,0.8459,0.0998,0.6777,0.8799,0.8443,0.2846,0.6064


,panel,config,model,variant,test_n,test_cases,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large,uncertainty_note
0,mec_nonfasting,BASE,WideDeep,raw,515,93,0.5515,0.8674,0.0949,0.5609,0.6962,0.8552,0.4699,0.9385,0.8304,0.5610,0.0877,0.8442,True,0.2970,Exploratory conditional PSU-resampling interva...
1,mec_nonfasting,BASE,WideDeep,calibrated,515,93,0.5515,0.8674,0.0945,0.5609,0.6962,0.8431,0.4500,0.9377,0.8203,0.5466,0.2233,1.0268,True,0.1884,Exploratory conditional PSU-resampling interva...
2,mec_nonfasting,BMI,WideDeep,raw,515,93,0.5231,0.8837,0.0959,0.5354,0.5999,0.8862,0.4929,0.9232,0.8416,0.5412,0.2550,0.8569,True,0.4660,Exploratory conditional PSU-resampling interva...
3,mec_nonfasting,BMI,WideDeep,calibrated,515,93,0.5231,0.8837,0.0941,0.5354,0.6763,0.8455,0.4465,0.9341,0.8191,0.5379,0.3628,1.1350,True,0.1884,Exploratory conditional PSU-resampling interva...
4,mec_nonfasting,WHR,WideDeep,raw,515,93,0.5845,0.8881,0.0896,0.5959,0.6618,0.8794,0.5028,0.9338,0.8455,0.5715,0.2675,0.8978,True,0.4047,Exploratory conditional PSU-resampling interva...
5,mec_nonfasting,WHR,WideDeep,calibrated,515,93,0.5845,0.8881,0.0898,0.5959,0.7600,0.8477,0.4791,0.9504,0.8340,0.5877,0.4546,1.1769,True,0.2313,Exploratory conditional PSU-resampling interva...
6,mec_nonfasting,BOTH,MLP,raw,515,93,0.5776,0.8805,0.0928,0.5871,0.6299,0.8739,0.4795,0.9276,0.8359,0.5445,0.0278,0.7450,True,0.3554,Exploratory conditional PSU-resampling interva...
7,mec_nonfasting,BOTH,MLP,calibrated,515,93,0.5776,0.8805,0.0923,0.5871,0.6299,0.8670,0.4660,0.9270,0.8300,0.5357,0.3906,1.0826,True,0.2803,Exploratory conditional PSU-resampling interva...
8,fasting_with_glucose,BASE,MLP,raw,215,44,0.8208,0.9292,0.0668,0.8247,0.7138,0.9325,0.6935,0.9383,0.8939,0.7035,0.5845,0.8799,True,0.8022,Exploratory conditional PSU-resampling interva...
9,fasting_with_glucose,BASE,MLP,calibrated,215,44,0.8208,0.9292,0.0658,0.8247,0.7138,0.9171,0.6483,0.9374,0.8813,0.6795,0.4552,1.0559,True,0.3767,Exploratory conditional PSU-resampling interva...


,panel,comparison,delta_auc,lower_psu,upper_psu,replicates
0,mec_nonfasting,BMI minus BASE,-0.0050,-0.0165,0.0080,1000
1,mec_nonfasting,WHR minus BASE,0.0013,-0.0074,0.0092,1000
2,mec_nonfasting,BOTH minus BASE,-0.0021,-0.0112,0.0041,1000
3,mec_nonfasting,WHR minus BMI,0.0063,-0.0034,0.0149,1000
4,mec_nonfasting,BOTH minus BMI,0.0029,-0.0039,0.0107,1000
5,fasting_with_glucose,BMI minus BASE,-0.0016,-0.0124,0.0064,1000
6,fasting_with_glucose,WHR minus BASE,0.0035,-0.0087,0.0215,1000
7,fasting_with_glucose,BOTH minus BASE,0.0015,-0.0046,0.0181,1000
8,fasting_with_glucose,WHR minus BMI,0.0051,-0.0087,0.0260,1000
9,fasting_with_glucose,BOTH minus BMI,0.0031,-0.0046,0.0221,1000


In [11]:
RUN_COMPLETE_NHANES_PIPELINE = False

if RUN_COMPLETE_NHANES_PIPELINE:
    assert len(sorted(raw_dir.glob("*.xpt"))) == 14, "Download all public NHANES files first"
    subprocess.run(
        [sys.executable, "-u", "run_all.py", "--analysis-only"],
        cwd=nhanes_dir,
        check=True,
    )
    # Refresh the result objects so every later cell checks this new run rather
    # than the packaged aggregate reference values loaded earlier.
    nhanes_results = live_outputs
    logistic_results = pd.read_csv(nhanes_results / "common_logistic_results.csv")
    neural_results = pd.read_csv(nhanes_results / "all_neural_test_results.csv")
    paired_results = pd.read_csv(nhanes_results / "paired_logistic_auc_differences.csv")
    print("PASS: complete NHANES pipeline finished.")
else:
    print("Complete pipeline skipped. Change RUN_COMPLETE_NHANES_PIPELINE to True for a full rerun.")


Complete pipeline skipped. Change RUN_COMPLETE_NHANES_PIPELINE to True for a full rerun.


In [12]:
borderline = pd.read_csv(nhanes_results / "borderline_conclusion_sensitivity.csv")
definite_test = pd.read_csv(nhanes_results / "borderline_models_on_definite_test.csv")
complete_case = pd.read_csv(nhanes_results / "complete_case_sensitivity.csv")
thresholds = pd.read_csv(nhanes_results / "mec_nonfasting" / "BOTH_LogReg_threshold_sensitivity.csv")
shap_importance = pd.read_csv(nhanes_results / "mec_nonfasting" / "BOTH_LogReg_shap_importance.csv")

display(borderline.round(4))
display(definite_test.round(4))
display(complete_case.round(4))
display(thresholds.round(4))
display(shap_importance.head(15).round(4))


,panel,scenario,delta_auc,lower_psu,upper_psu,interpretation
0,mec_nonfasting,borderline excluded,0.0063,-0.0034,0.0149,interval includes zero
1,mec_nonfasting,borderline non-positive,0.0080,-0.0005,0.0167,interval includes zero
2,fasting_with_glucose,borderline excluded,0.0051,-0.0087,0.0260,interval includes zero
3,fasting_with_glucose,borderline non-positive,0.0037,-0.0078,0.0184,interval includes zero
4,fasting_without_glucose,borderline excluded,0.0100,-0.0010,0.0212,interval includes zero
5,fasting_without_glucose,borderline non-positive,0.0106,-0.0056,0.0274,interval includes zero


,panel,config,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large
0,mec_nonfasting,BASE,0.5792,0.8906,0.0890,0.5919,0.7020,0.8708,0.5004,0.9407,0.8445,0.5843,0.0860,0.8894,True,0.2234
1,mec_nonfasting,BMI,0.5575,0.8822,0.0895,0.5689,0.7001,0.8738,0.5056,0.9405,0.8467,0.5872,0.1019,0.8853,True,0.2474
2,mec_nonfasting,WHR,0.6163,0.8909,0.0855,0.6273,0.6566,0.8824,0.5073,0.9331,0.8473,0.5724,0.1119,0.8936,True,0.2450
3,mec_nonfasting,BOTH,0.6193,0.8856,0.0866,0.6304,0.6533,0.8929,0.5293,0.9332,0.8556,0.5848,0.0914,0.8727,True,0.2518
4,fasting_with_glucose,BASE,0.7879,0.9032,0.0714,0.7918,0.7316,0.8897,0.5867,0.9393,0.8618,0.6512,-0.1225,0.7934,True,0.1290
5,fasting_with_glucose,BMI,0.7856,0.9051,0.0724,0.7897,0.7407,0.8972,0.6067,0.9417,0.8696,0.6670,-0.0759,0.7498,True,0.2575
6,fasting_with_glucose,WHR,0.7945,0.9089,0.0693,0.7984,0.7245,0.8698,0.5438,0.9365,0.8442,0.6213,0.0085,0.8704,True,0.1730
7,fasting_with_glucose,BOTH,0.7906,0.9067,0.0695,0.7947,0.7245,0.8858,0.5759,0.9376,0.8573,0.6417,0.0147,0.8472,True,0.2138
8,fasting_without_glucose,BASE,0.5857,0.8272,0.1000,0.6076,0.7140,0.8633,0.5280,0.9338,0.8370,0.6071,-0.1390,0.6672,True,0.2481
9,fasting_without_glucose,BMI,0.6374,0.8458,0.0986,0.6521,0.6777,0.8799,0.5472,0.9273,0.8443,0.6055,-0.1301,0.6161,True,0.3468


,panel,config,cohort_n,train_n,validation_n,test_n,test_cases,status,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1,calibration_intercept_joint,calibration_slope,calibration_fit_converged,calibration_in_the_large
0,mec_nonfasting,BASE,1288,759,275,254,35,Exploratory; changed population and potentiall...,0.5504,0.8931,0.0750,0.6091,0.7032,0.8688,0.4608,0.9483,0.8460,0.5567,0.0796,1.0476,True,0.0253
1,mec_nonfasting,BMI,1288,759,275,254,35,Exploratory; changed population and potentiall...,0.5565,0.8893,0.0747,0.6094,0.7032,0.8959,0.5184,0.9498,0.8694,0.5968,0.1065,1.0518,True,0.0465
2,mec_nonfasting,WHR,1288,759,275,254,35,Exploratory; changed population and potentiall...,0.5916,0.8761,0.0781,0.6224,0.7032,0.9211,0.5869,0.9511,0.8911,0.6398,0.3600,1.1345,True,0.1743
3,mec_nonfasting,BOTH,1288,759,275,254,35,Exploratory; changed population and potentiall...,0.5883,0.8766,0.0777,0.6225,0.7032,0.9230,0.5928,0.9512,0.8928,0.6433,0.3561,1.1301,True,0.1764
4,fasting_with_glucose,BASE,589,353,112,124,17,Exploratory; changed population and potentiall...,0.6935,0.8406,0.0588,0.6997,0.6752,0.9351,0.5824,0.9555,0.9043,0.6254,-0.4895,0.6796,True,-0.0488
5,fasting_with_glucose,BMI,589,353,112,124,17,Exploratory; changed population and potentiall...,0.6873,0.8320,0.0587,0.6933,0.6955,0.9351,0.5896,0.9582,0.9067,0.6382,-0.3634,0.7083,True,0.0638
6,fasting_with_glucose,WHR,589,353,112,124,17,Exploratory; changed population and potentiall...,0.6952,0.8496,0.0593,0.7015,0.6955,0.9484,0.6440,0.9587,0.9185,0.6688,-0.3800,0.7614,True,-0.0672
7,fasting_with_glucose,BOTH,589,353,112,124,17,Exploratory; changed population and potentiall...,0.6960,0.8502,0.0593,0.7023,0.6955,0.9484,0.6440,0.9587,0.9185,0.6688,-0.3684,0.7604,True,-0.0507
8,fasting_without_glucose,BASE,589,353,112,124,17,Exploratory; changed population and potentiall...,0.3325,0.7584,0.0946,0.4001,0.6955,0.7961,0.3139,0.9512,0.7842,0.4326,-0.8299,0.5633,True,-0.2933
9,fasting_without_glucose,BMI,589,353,112,124,17,Exploratory; changed population and potentiall...,0.3536,0.7527,0.0899,0.4229,0.6955,0.8347,0.3607,0.9534,0.8182,0.4751,-0.7891,0.5595,True,-0.2104


,threshold,pr_auc,auc,brier,average_precision,sensitivity,specificity,ppv,npv,accuracy,f1
0,0.1,0.6211,0.8863,0.0866,0.6322,0.8031,0.7758,0.3977,0.9553,0.7800,0.5320
1,0.2,0.6211,0.8863,0.0866,0.6322,0.6646,0.8837,0.5130,0.9346,0.8496,0.5790
2,0.3,0.6211,0.8863,0.0866,0.6322,0.5945,0.9089,0.5460,0.9240,0.8599,0.5692
3,0.5,0.6211,0.8863,0.0866,0.6322,0.4873,0.9571,0.6770,0.9101,0.8840,0.5667


,feature,weighted_mean_absolute_shap
0,age,0.7591
1,family_history_diabetes,0.6413
2,waist_to_hip_ratio,0.6298
3,hypertension_history,0.4362
4,heart_rate,0.4330
5,education_level,0.3803
6,hdl_cholesterol,0.2959
7,ethnicity,0.2902
8,diastolic_bp,0.2493
9,cholesterol_total,0.2087


In [13]:
metric_verification = json.loads(
    (nhanes_results / "metric_recalculation.json").read_text(encoding="utf-8")
)
resampling_verification = json.loads(
    (nhanes_results / "resampling_endpoint_verification.json").read_text(encoding="utf-8")
)
complete_verification = json.loads(
    (nhanes_results / "complete_verification.json").read_text(encoding="utf-8")
)

print("Metric recalculation passed:", metric_verification["passed"])
print("Metric values checked:", metric_verification.get("count"))
print("Resampling verification passed:", resampling_verification["passed"])
print("Resampling endpoint pairs checked:", len(resampling_verification.get("checks", [])))

assert metric_verification["passed"] is True
assert resampling_verification["passed"] is True
display(complete_verification)


Metric recalculation passed: True
Metric values checked: 3392
Resampling verification passed: True
Resampling endpoint pairs checked: 1486


{'checks': [{'panel': 'mec_nonfasting',
   'psu_overlap': 0,
   'classical_metric_rows_verified': 14,
   'neural_metric_rows_verified': 8,
   'neural_development_fits': 60,
   'final_neural_fits': 4},
  {'panel': 'fasting_with_glucose',
   'psu_overlap': 0,
   'classical_metric_rows_verified': 8,
   'neural_metric_rows_verified': 8,
   'neural_development_fits': 60,
   'final_neural_fits': 4},
  {'panel': 'fasting_without_glucose',
   'psu_overlap': 0,
   'classical_metric_rows_verified': 8,
   'neural_metric_rows_verified': 8,
   'neural_development_fits': 60,
   'final_neural_fits': 4},
  {'panel': 'mec_nonfasting_borderline_nonpositive',
   'psu_overlap': 0,
   'classical_metric_rows_verified': 14,
   'neural_metric_rows_verified': 8,
   'neural_development_fits': 60,
   'final_neural_fits': 4},
  {'panel': 'fasting_with_glucose_borderline_nonpositive',
   'psu_overlap': 0,
   'classical_metric_rows_verified': 10,
   'neural_metric_rows_verified': 8,
   'neural_development_fits': 60

In [14]:
primary_both = logistic_results.query(
    "panel == 'mec_nonfasting' and config == 'BOTH'"
).iloc[0]

assert round(primary_both["auc"], 4) == 0.8863
assert round(primary_both["brier"], 4) == 0.0866
assert round(primary_both["sensitivity"], 4) == 0.6646
assert round(primary_both["specificity"], 4) == 0.8837
assert round(primary_both["calibration_slope"], 4) == 0.8718

whr_minus_bmi = paired_results.query(
    "panel == 'mec_nonfasting' and comparison == 'WHR minus BMI'"
).iloc[0]

assert round(whr_minus_bmi["delta_auc"], 4) == 0.0063
assert round(whr_minus_bmi["lower_psu"], 4) == -0.0034
assert round(whr_minus_bmi["upper_psu"], 4) == 0.0149

print("PASS: principal NHANES values match the manuscript.")


PASS: principal NHANES values match the manuscript.
